In [1]:
import pandas as pd
import numpy as np
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids_aa = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_aa_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

X_f32 = X_auto_aa_int.astype(np.float32)
p_aa = X_f32.mean(axis=1) / 2
denom_aa = np.sqrt(2 * p_aa * (1 - p_aa))
valid_mask_aa = denom_aa > 1e-8
del X_f32
gc.collect()

valid_indices = np.where(valid_mask_aa)[0]
np.random.seed(0)
chosen_idx = np.random.RandomState(0).choice(valid_indices, 2000, replace=False)  # reduced to 2000

X_subset_int = X_auto_aa_int[chosen_idx].astype(np.float64)
del X_auto_aa_int
gc.collect()

p_subset = p_aa[chosen_idx]
denom_subset = denom_aa[chosen_idx]
X_subset_std = ((X_subset_int - 2 * p_subset[:, None]) / denom_subset[:, None]).T
del X_subset_int
gc.collect()

print("Subset shape:", X_subset_std.shape)

sample_corr_aa = np.corrcoef(X_subset_std)
del X_subset_std
gc.collect()
np.fill_diagonal(sample_corr_aa, 0)

max_pairwise_corr_aa = sample_corr_aa.max(axis=1)
print(f"Samples with max pairwise corr > 0.5: {(max_pairwise_corr_aa > 0.5).sum()}")
print(f"Samples with max pairwise corr > 0.3: {(max_pairwise_corr_aa > 0.3).sum()}")

# VECTORIZED pair extraction (no Python loop) using upper triangle
iu = np.triu_indices_from(sample_corr_aa, k=1)
corr_values_flat = sample_corr_aa[iu]
strong_mask = corr_values_flat > 0.3
strong_corr_values = corr_values_flat[strong_mask]

print(f"\nTotal pairs found: {strong_mask.sum()}")
print(f"  >0.9: {(strong_corr_values > 0.9).sum()}")
print(f"  0.5-0.9: {((strong_corr_values >= 0.5) & (strong_corr_values <= 0.9)).sum()}")
print(f"  0.3-0.5: {((strong_corr_values >= 0.3) & (strong_corr_values < 0.5)).sum()}")

Subset shape: (3348, 2000)
Samples with max pairwise corr > 0.5: 745
Samples with max pairwise corr > 0.3: 1790

Total pairs found: 1736
  >0.9: 229
  0.5-0.9: 250
  0.3-0.5: 1257
